# holdout-data-one-per-class — worked example 1: Holdout gallery: first sample per class from a shuffled batch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `holdout-data-one-per-class`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A holdout gallery is constructed by scanning a dataset and keeping exactly one representative example per class. The pattern is: iterate class indices 0..K-1, apply a boolean mask `labels == c` to select all samples of that class, then take index [0] from the masked selection. Stacking the resulting list produces a tensor of shape `(num_classes, *sample_shape)` indexed by class.

## Worked solution

**Step 1 — create synthetic data.** We simulate a shuffled batch of 50 samples across 5 classes with feature dimension 8. Labels are randomly permuted so the first sample of each class is not at a predictable position.

**Step 2 — apply the mask pattern.** For each class `c`, `labels == c` produces a boolean tensor. `data[mask][0]` selects all samples of class `c` and takes the first one.

**Step 3 — collect and stack.** Append each representative to a list, then `t.stack(per_class, dim=0)`. The output shape is `(5, 8)`.

**Step 4 — verify shape and class correctness.** Confirm the output shape is `(num_classes, 8)`. Also verify that the holdout for class `c` actually belongs to class `c` (round-trip check via the original labels tensor).

In [ ]:
import torch as t

t.manual_seed(10)

def one_per_class(data: t.Tensor, labels: t.Tensor, num_classes: int) -> t.Tensor:
    per_class = []
    for c in range(num_classes):
        mask = labels == c
        per_class.append(data[mask][0])
    return t.stack(per_class, dim=0)

# Create synthetic data: 50 samples, 5 classes, 8 features
t.manual_seed(10)
N, C, D = 50, 5, 8
data = t.randn(N, D)
# Make sure every class appears: assign labels in round-robin then shuffle
base_labels = t.arange(N) % C
perm = t.randperm(N)
data = data[perm]
labels = base_labels[perm]

holdout = one_per_class(data, labels, C)
print(f"Holdout shape: {holdout.shape}")  # (5, 8)
assert holdout.shape == (C, D)

# Verify: for class c, the holdout must come from an actual class-c sample
for c in range(C):
    # Find the holdout in the original data
    match = (data == holdout[c]).all(dim=1).nonzero(as_tuple=True)[0]
    assert len(match) >= 1, f"Class {c} holdout not found in data"
    assert labels[match[0]].item() == c, f"Class {c} holdout has wrong label"

print("Shape and class-membership checks passed.")